<a href="https://colab.research.google.com/github/elsarsa/esktraksi-fitur-citra-lombok/blob/main/2418018_ElsarHendriFiturwarnateskturbentuk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
from google.colab import drive

# 1. Alamat Folder (PASTIKAN SUDAH BENAR)
# Klik kanan folder di kiri Colab -> Copy Path
folder_path = '/content/drive/MyDrive/dataset_lombok'

data_fitur = []

# Cek apakah folder terbaca
if not os.path.exists(folder_path):
    print("Folder tidak ditemukan")
else:
    # Ambil daftar file (Mendukung .jpg .JPG .png .PNG)
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Ditemukan {len(files)} citra. Memulai proses...")

    for file_name in sorted(files):
        path = os.path.join(folder_path, file_name)
        img = cv2.imread(path)

        if img is None:
            print(f"Gagal membaca: {file_name}")
            continue

        # --- PREPROCESSING ---
        img_res = cv2.resize(img, (500, 500))
        hsv = cv2.cvtColor(img_res, cv2.COLOR_BGR2HSV)
        gray = cv2.cvtColor(img_res, cv2.COLOR_BGR2GRAY)

        # FITUR WARNA (HSV)
        # Mengambil rata-rata Hue, Saturation, Value
        h, s, v = cv2.split(hsv)
        mean_h = np.mean(h)
        mean_s = np.mean(s)
        mean_v = np.mean(v)

        # FITUR TEKSTUR (GLCM)
        #  Contrast, Correlation, Energy, Homogeneity
        glcm = graycomatrix(gray, [1], [0], symmetric=True, normed=True)
        contrast = graycoprops(glcm, 'contrast')[0, 0]
        correlation = graycoprops(glcm, 'correlation')[0, 0]
        energy = graycoprops(glcm, 'energy')[0, 0]
        homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]

        # FITUR BENTUK (Morfologi/Threshold)
        # Binerisasi untuk memisahkan lombok dari background putih
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            cnt = max(contours, key=cv2.contourArea)
            area = cv2.contourArea(cnt)
            perimeter = cv2.arcLength(cnt, True)
            # Metric (Circularity)
            metric = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
        else:
            area, perimeter, metric = 0, 0, 0


        data_fitur.append([
            file_name, mean_h, mean_s, mean_v,
            contrast, correlation, energy, homogeneity,
            area, perimeter, metric
        ])
        print(f"Berhasil: {file_name}")

    #  MEMBUAT TABEL
    kolom = [
        'Foto', 'H', 'S', 'V',
        'Kontras', 'Korelasi', 'Energi', 'Homogenitas',
        'Area', 'Perimeter', 'Metric'
    ]
    df = pd.DataFrame(data_fitur, columns=kolom)

    # Simpan ke CSV
    df.to_csv('Hasil_Ekstraksi_Lombok.csv', index=False)

    print("\n--- PROSES SELESAI ---")
    # Menampilkan tabel di Colab
    from google.colab import data_table
    display(data_table.DataTable(df, include_index=False, num_rows_per_page=10))

Ditemukan 10 citra. Memulai proses...
Berhasil: Citra1.jpg
Berhasil: Citra10.jpg
Berhasil: Citra2.jpg
Berhasil: Citra3.jpg
Berhasil: Citra4.jpg
Berhasil: Citra5.jpg
Berhasil: Citra6.jpg
Berhasil: Citra7.jpg
Berhasil: Citra8.jpg
Berhasil: Citra9.jpg

--- PROSES SELESAI ---


,Foto,H,S,V,Kontras,Korelasi,Energi,Homogenitas,Area,Perimeter,Metric
0,Citra1.jpg,111.305720,15.231048,119.597088,5.288136,0.998552,0.073052,0.799323,95718.0,2265.190030,0.234420
1,Citra10.jpg,55.653080,11.343608,156.576544,4.959427,0.997991,0.082059,0.809229,48601.5,1448.516806,0.291080
2,Citra2.jpg,102.445260,14.617840,169.574876,3.335623,0.999074,0.079742,0.797402,59237.5,2281.300638,0.143035
3,Citra3.jpg,124.313508,15.679384,160.489520,4.983467,0.998403,0.083644,0.801620,47725.0,1447.587874,0.286198
4,Citra4.jpg,129.863868,15.679848,142.263668,5.239960,0.998763,0.070060,0.793587,80534.0,2289.248904,0.193109
5,Citra5.jpg,102.059440,14.682624,187.158372,4.634858,0.998759,0.085566,0.807715,55008.0,2305.685415,0.130028
6,Citra6.jpg,113.248696,14.770904,141.775684,5.241271,0.998764,0.069939,0.793159,80558.5,2291.148399,0.192848
7,Citra7.jpg,102.422544,14.620620,169.575028,3.335479,0.999074,0.079751,0.797418,59238.5,2281.300638,0.143037
8,Citra8.jpg,99.200280,20.089940,159.782996,4.961743,0.998005,0.081759,0.807602,49225.0,1451.872145,0.293453
9,Citra9.jpg,102.084640,15.395676,169.197288,3.337335,0.999068,0.079140,0.796189,60313.5,2275.643783,0.146358
